Load libraries

In [ ]:
# the tv env works on this, but use the specific conda env for this:
# conda activate scrna-downstream
import scanpy as sc
import decoupler as dc
import numpy as np
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import gseapy as gp
import seaborn as sns
from scipy.stats import zscore
from matplotlib.backends.backend_pdf import PdfPages
import pickle

run = False # for certain blocks, like the excel file and pdf file blocks, set to True if they need to be run


In this file, we will perform certain downstream analyses: pseudobulk analysis, conserved vs specific markers per treatment, gene modules, pathwat enrichment, GRN

In [ ]:
adata = sc.read_h5ad("c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Annotation_Comparison/final_annotation_2026.h5ad")

In [ ]:
adata

In [ ]:
# this .obs column is the right one for the annotations
adata.obs["celltype_final_2026"].value_counts(dropna=False)

## Pseudobulk

This groups cells together based on the group you specify, collapsing the counts into one profile. Then you can compare these different groups. Then you can do DE analysis between the "true replicates", which are the mice from which the cells of one sample came. 

In [ ]:
adata

In [ ]:
adata.obs["LENIENT_GUESS_merge"].value_counts()

In [ ]:
# hashtags appear over multiple experiments, because they were re-used
pd.crosstab(adata.obs["orig.ident"], adata.obs["LENIENT_GUESS_merge"])

In [ ]:
# 1. Clean the hashtag column by removing everything up to the second underscore
hashtag_raw = adata.obs["LENIENT_GUESS_merge"].astype(str)
clean_hashtag = hashtag_raw.str.split("_", n=2).str[-1]

# 2. Extract sample ID
sample_col = adata.obs["orig.ident"].astype(str)

# 3. Create the combined name (Hashtag_SampleID)
combined_name = clean_hashtag + "_" + sample_col

# 4. Identify cells where the hashtag is NA / missing
is_na_mask = (
    adata.obs["LENIENT_GUESS_merge"].isna()
    | (adata.obs["LENIENT_GUESS_merge"] == "NA")
    | (clean_hashtag == "nan")
)

# 5. If NA, use orig.ident; otherwise, use combined_name
adata.obs["mouse_id"] = np.where(is_na_mask, sample_col, combined_name)

In [ ]:
adata.obs["mouse_id"].value_counts()
# this is correct

In [ ]:
adata.obs["experiment"].unique()

SAM6 needs to be WT, was a mistake

In [ ]:
# sanity check, to verify the treatment cols
check = pd.crosstab(
    [adata.obs["experiment"], adata.obs["celltype_final_2026"]],
    adata.obs["treatment"]
)

print(check)

In [ ]:
# sanity check, to verify the treatment cols
missing = check[(check.get("WT", 0) == 0) | (check.get("Test", 0) == 0)]

print(missing)

In [ ]:
adata.obs.loc[
    adata.obs["orig.ident"] == "SAM06",
    "treatment"
] = "WT"

In [ ]:
# .X layer is lognormalized, so we need to use the "counts" layer
# pseudobulk is done on individual mice and celltypes
pdata = dc.pp.pseudobulk(
    adata=adata,
    sample_col="mouse_id", # individual mice
    groups_col="celltype_final_2026", # for each mouse, on cluster level
    mode="sum",
    layer="counts"
)

# this block creates one pseudobulk profile for every cell type present within each individual mouse

In [ ]:
dc.pl.filter_samples(
    adata=pdata,
    groupby=["mouse_id"],
    min_cells=10,
    min_counts=1000,
    figsize=(10, 10),
)

# each dot is a unique combination of sample and celltype (15 x 9 dots for these specifics)

In [ ]:
dc.pp.filter_samples(pdata, min_cells=10, min_counts=1000)

In [ ]:
dc.pl.obsbar(adata=pdata, y="celltype_final_2026", hue="mouse_id", figsize=(10, 10))

Variability exploration.

In [ ]:
# the data is not normalized in pdata
pdata.X.max()

In [ ]:
pdata.X.min()

In [ ]:
# Store raw counts in layers
pdata.layers["counts"] = pdata.X.copy()

# Normalize, scale and compute pca
# PCA requires this
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)

# Return raw counts to X
dc.pp.swap_layer(adata=pdata, key="counts", inplace=True)

calculate associations between each inferred PC and the variables in the metadata

In [ ]:
dc.tl.rankby_obsm(pdata, key="X_pca")

In [ ]:
sc.pl.pca_variance_ratio(pdata)

In [ ]:
sc.pl.pca(
    pdata,
    color=["mouse_id", "celltype_final_2026"],
    ncols=1,
    size=300,
    frameon=True,
)

Feature selection

In [ ]:
dc.pl.filter_by_expr(
    adata=pdata,
    group="celltype_final_2026",
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)
dc.pl.filter_by_prop(
    adata=pdata,
    min_prop=0.1,
    min_smpls=2,
)

# in the plots below, only genes in the top right quadrant of the first plot are retained
# in the bottom plot, only genes right of the dashed line are kept

In [ ]:
# here, actual filtering takes place

dc.pp.filter_by_expr(
    adata=pdata,
    group="mouse_id",
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)
dc.pp.filter_by_prop( # maybe be more restrictive here in the future
    adata=pdata,
    min_prop=0.1,
    min_smpls=2,
)
pdata

# 26000 features have been removed using this filtering 

DE analysis on the pseudobulk, we will do it for following comparisons: 
- Test vs WT in each separate experiment
- Toxo vs other immunogenic conditions

In [ ]:
pdata

In [ ]:
pdata.obs["experiment"].unique()

Do DE on Test vs WT in each experiment. The DE results are stored for each celltype. 

NOTE: CITEseq_Toxo and CITEseq_Final hold both "Test" and "WT" cells, while all other experiments only hold one of the 2. This is a problem in the DEseq analysis, since we cannot specify the 2 groups within one experiment. Is CITEseq_LNP_WT the shared control group for all different LNP formulations?

So for Toxo and Final, we compar within the group based on treatment.
For the LNP groups, we compare each test group against the WT LNP group. 

In [ ]:
experiment_table = (
    adata.obs
    .groupby("experiment")
    .agg(
        samples=("orig.ident", lambda x: ", ".join(sorted(x.unique()))),
        treatments=("treatment", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

experiment_table

In [ ]:
# CITEseq_Toxo and CITEseq_Final DEseq2, as they hold the 2 treatment groups
inference = DefaultInference(n_cpus=8)

results_toxo_final = {}

for exp in ["CITEseq_Toxo"]:
    print(f"====================={exp}=====================")

    pdata_exp = pdata[pdata.obs["experiment"] == exp].copy()

    for ct in pdata_exp.obs["celltype_final_2026"].unique():

        sub = pdata_exp[
            pdata_exp.obs["celltype_final_2026"] == ct
        ].copy()

        # Require WT and Test samples
        if not {"WT", "Test"}.issubset(sub.obs["treatment"].unique()):
            continue

        # Require biological replicates
        reps = sub.obs.groupby("treatment")["mouse_id"].nunique()
        if (reps < 2).any():
            continue

        dds = DeseqDataSet(
            adata=sub,
            design="~treatment",
            refit_cooks=True,
        )

        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["treatment", "Test", "WT"],
            inference=inference,
        )

        stat_res.summary() # do not unhash this, because for some reason, this causes to an error on the following line if this line is not executed
        results_toxo_final[(exp, ct)] = stat_res.results_df


In [ ]:
results_toxo_final[("CITEseq_Toxo", "Late Mature")]

In [ ]:
summary = []

for (comparison, celltype), df in results_toxo_final.items():

    sig = df[df["padj"] < 0.05]

    summary.append({
        "comparison": comparison,
        "celltype": celltype,
        "total_genes": df.shape[0],
        "DE_genes": len(sig),
        "up": (sig["log2FoldChange"] > 0).sum(),
        "down": (sig["log2FoldChange"] < 0).sum()
    })

summary = pd.DataFrame(summary)
summary

# cell number has influence on how many DE

In [ ]:
plot_df = summary.pivot(
    index="celltype",
    columns="comparison",
    values="DE_genes"
)

plt.figure(figsize=(10,8))

sns.heatmap(
    plot_df,
    annot=True,
    fmt=".0f"
)

plt.title("Number of DE genes")
plt.show()

In [ ]:
# identify top genes 
top_genes_toxo_final = set()

for key, res in results_toxo_final.items():

    sig = res[res["padj"] < 0.05]

    top = (
        sig
        .sort_values("stat", ascending=False) #set to True for downregulated genes
        .head(20)
        .index
    )

    top_genes_toxo_final.update(top)

top_genes_toxo_final = list(top_genes_toxo_final)
top_genes_toxo_final

DEseq2 for the LNP groups

In [ ]:
inference = DefaultInference(n_cpus=8)

results_lnp = {}

lnp_comparisons = {
    "CITEseq_LNP_CpG_LNPs": "CITEseq_LNP_WT",
    "CITEseq_LNP_eLNPs": "CITEseq_LNP_WT",
    "CITEseq_LNP_pIC_LNPs": "CITEseq_LNP_WT",
    "CITEseq_LNP_pIC": "CITEseq_LNP_WT"
}

for test_exp, wt_exp in lnp_comparisons.items():
    print(f"====================={test_exp}=====================")

    pdata_lnp = pdata[
        pdata.obs["experiment"].isin(
            [test_exp, wt_exp]
        )
    ].copy()

    for ct in pdata_lnp.obs["celltype_final_2026"].unique():

        sub = pdata_lnp[
            pdata_lnp.obs["celltype_final_2026"] == ct
        ].copy()

        # Require both groups
        if not {test_exp, wt_exp}.issubset(
            sub.obs["experiment"].unique()
        ):
            continue

        # Require biological replicates
        reps = sub.obs.groupby("experiment")["mouse_id"].nunique()
        if (reps < 2).any():
            continue

        dds = DeseqDataSet(
            adata=sub,
            design="~experiment",
            refit_cooks=True,
        )

        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=[
                "experiment",
                test_exp,
                wt_exp
            ],
            inference=inference,
        )

        stat_res.summary()
        results_lnp[(test_exp, ct)] = stat_res.results_df

In [ ]:
# results of LNP, specified per celltype
results_lnp[("CITEseq_LNP_CpG_LNPs", "Late Mature")].sort_values(
    by="log2FoldChange",
    ascending=False # False means highest LFC is at the top, True means most downregulted is at the top
)

# "stat" is the Wald test statistic, meaning how strongly the observed LFC differs from 0, taking the standard error into account
# it is essentially the signal to noise ratio


In [ ]:
results_lnp

In [ ]:
# possible to plot specific genes for a specific cell group in this comparison
res = results_lnp[("CITEseq_LNP_CpG_LNPs", "Late Mature")]
res.loc[["Xcr1", "Clec9a", "Batf3"]]

In [ ]:
summary = []

for (comparison, celltype), df in results_lnp.items():

    sig = df[df["padj"] < 0.05]

    summary.append({
        "comparison": comparison,
        "celltype": celltype,
        "total_genes": df.shape[0],
        "DE_genes": len(sig),
        "up": (sig["log2FoldChange"] > 1).sum(),
        "down": (sig["log2FoldChange"] < -1).sum()
    })

summary = pd.DataFrame(summary)
summary

In [ ]:
# VOLCANO PLOT FOR RESULTS
res = results_lnp[("CITEseq_LNP_CpG_LNPs", "Late Mature")].copy()

res["-log10_padj"] = -np.log10(res["padj"])

plt.figure(figsize=(7,6))

plt.scatter(
    res["log2FoldChange"],
    res["-log10_padj"],
    s=5
)

plt.axvline(1, linestyle="--")
plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 adjusted p-value")
plt.title("CpG LNP vs LNP WT - Late Mature")

plt.show()

In [ ]:
summary = []

for (comparison, celltype), res in results_lnp.items():

    sig = res[res["padj"] < 0.05]

    summary.append({
        "comparison": comparison,
        "celltype": celltype,
        "DE_genes": len(sig)
    })

summary = pd.DataFrame(summary)


plot_df = summary.pivot(
    index="celltype",
    columns="comparison",
    values="DE_genes"
)

plt.figure(figsize=(10,8))

sns.heatmap(
    plot_df,
    annot=True,
    fmt=".0f"
)

plt.title("Number of DE genes")
plt.show()

In [ ]:
# identify top genes 
top_genes_lnp = set()

for key, res in results_lnp.items():

    sig = res[res["padj"] < 0.05]

    top = (
        sig
        .sort_values("stat", ascending=False) #set to True for downregulated genes
        .head(20)
        .index
    )

    top_genes_lnp.update(top)

top_genes_lnp = list(top_genes_lnp)
top_genes_lnp

Further DE analysis, with Toxo against other "Test" treatments of each experiment, and the CITEseq_Test WT as baseline.

In [ ]:
adata.obs["experiment"].value_counts()

In [ ]:
# toxo Test versus all other immunogenic treatments
inference = DefaultInference(n_cpus=8)

results_toxo_vs_other = {}

comparisons = {
    "CITEseq_LNP_CpG_LNPs": "CITEseq_LNP_CpG_LNPs",
    "CITEseq_LNP_eLNPs": "CITEseq_LNP_eLNPs",
    "CITEseq_LNP_pIC": "CITEseq_LNP_pIC",
    "CITEseq_LNP_pIC_LNPs": "CITEseq_LNP_pIC_LNPs",
}

for test_exp in comparisons.keys():

    # Select Toxo Test + comparison experiment Test
    # subset, using the "Test" from Toxo and the "Test" for each other experiment, specified before
    sub = pdata[
        (
            (pdata.obs["experiment"] == "CITEseq_Toxo") &
            (pdata.obs["treatment"] == "Test")
        )
        |
        (
            (pdata.obs["experiment"] == test_exp) &
            (pdata.obs["treatment"] == "Test")
        )
    ].copy()
    print(f"=========================={test_exp}=========================")

    for ct in sub.obs["celltype_final_2026"].unique():

        cell = sub[
            sub.obs["celltype_final_2026"] == ct
        ].copy()


        # Require both experiments
        if not {"CITEseq_Toxo", test_exp}.issubset(
            cell.obs["experiment"].unique()
        ):
            continue


        # Require biological replicates
        reps = (
            cell.obs
            .groupby("experiment")["mouse_id"]
            .nunique()
        )

        if (reps < 2).any():
            continue


        dds = DeseqDataSet(
            adata=cell,
            design="~experiment", # design is based on experiment, could also add other predictors
            refit_cooks=True,
        )

        dds.deseq2()


        stat_res = DeseqStats(
            dds,
            contrast=[ # contrast between the 2 different treatments
                "experiment",
                "CITEseq_Toxo",
                test_exp
            ],
            inference=inference,
        )

        stat_res.summary()


        results_toxo_vs_other[(test_exp, ct)] = stat_res.results_df

In [ ]:
# results
results_toxo_vs_other[("CITEseq_LNP_CpG_LNPs", "Early Mature")]

In [ ]:
summary = []

for (comparison, celltype), df in results_toxo_vs_other.items():

    sig = df[df["padj"] < 0.05]

    summary.append({
        "comparison": comparison,
        "celltype": celltype,
        "total_genes": df.shape[0],
        "DE_genes": len(sig),
        "up": (sig["log2FoldChange"] > 0).sum(),
        "down": (sig["log2FoldChange"] < 0).sum()
    })

summary = pd.DataFrame(summary)
summary

In [ ]:
plot_df = summary.pivot(
    index="celltype",
    columns="comparison",
    values="DE_genes"
)

plt.figure(figsize=(10,8))

sns.heatmap(
    plot_df,
    annot=True,
    fmt=".0f"
)

plt.title("Number of DE genes")
plt.show()

In [ ]:
# identify top genes 
top_genes_toxo_vs_other = set()

for key, res in results_toxo_vs_other.items():

    sig = res[res["padj"] < 0.05]

    top = (
        sig
        .sort_values("stat", ascending=False) #set to True for downregulated genes
        .head(20)
        .index
    )

    top_genes_toxo_vs_other.update(top)

top_genes_toxo_vs_other = list(top_genes_toxo_vs_other)
top_genes_toxo_vs_other

In [ ]:
# Finally, write out all DESeq2 results for the RShiny tool later

# for results lnp
with pd.ExcelWriter("DESeq2/DESeq2_results_LNP.xlsx", engine="openpyxl") as writer:

    for (experiment, celltype), df in results_lnp.items():

        sheet_name = f"{experiment[:18]}_{celltype}" 

        # Remove characters not allowed in Excel sheet names
        for char in ['\\', '/', '*', '?', ':', '[', ']']:
            sheet_name = sheet_name.replace(char, "_")

        sheet_name = sheet_name[:31] # excel sheets have a 31 character limit

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=True
        )

# for results toxo vs other
with pd.ExcelWriter("DESeq2/DESeq2_results_Toxo_vs_other.xlsx", engine="openpyxl") as writer:

    for (experiment, celltype), df in results_toxo_vs_other.items():

        sheet_name = f"{experiment[:18]}_{celltype}"

        for char in ['\\', '/', '*', '?', ':', '[', ']']:
            sheet_name = sheet_name.replace(char, "_")

        sheet_name = sheet_name[:31]

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=True
        )

# results toxo test vs WT
with pd.ExcelWriter("DESeq2/DESeq2_results_Toxo_test_wt.xlsx", engine="openpyxl") as writer:

    for (experiment, celltype), df in results_toxo_final.items():

        sheet_name = f"{experiment[:18]}_{celltype}" 

        for char in ['\\', '/', '*', '?', ':', '[', ']']:
            sheet_name = sheet_name.replace(char, "_")

        sheet_name = sheet_name[:31]

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=True
        )

Find marker genes using a wilcoxon rank test, and then finding the intersect

In [ ]:
if run:    
# 1. Run 1-vs-rest differential expression across all cell types
    sc.tl.rank_genes_groups(
        adata,
        groupby="celltype_final_2026",  # Change to your cell-type column name
        method="wilcoxon",
        key_added="celltype_markers"
    )

    # Visualize marker genes per cell type
    sc.pl.rank_genes_groups(adata, n_genes=10, key="celltype_markers")

In [ ]:
if run:    
# pandas was causing problems in the next block, which is why this is added
    if not hasattr(pd.DataFrame, "append"):
        def _df_append(self, other, ignore_index=False, verify_integrity=False, sort=False):
            return pd.concat([self, other], ignore_index=ignore_index,
                            verify_integrity=verify_integrity, sort=sort)
        pd.DataFrame.append = _df_append

In [ ]:
if run:    
# Storage dictionary for enrichment results
    enrichment_results = {}

    cell_types = adata.obs["celltype_final_2026"].unique()

    for ct in cell_types:
        # Extract DE genes for this specific cell type
        de_df = sc.get.rank_genes_groups_df(adata, group=ct, key="celltype_markers")
        
        # Filter for top significant marker genes
        sig_genes = de_df[
            (de_df["pvals_adj"] < 0.05) & 
            (de_df["logfoldchanges"] >= 0.5)
        ]["names"].tolist()
        
        # Format gene symbols to title case (e.g., 'Cd8a')
        sig_genes = [g.capitalize() for g in sig_genes]
        
        if len(sig_genes) < 5:
            print(f"Skipping {ct}: Not enough significant marker genes found.")
            continue

        # Run Enrichr query
        enr = gp.enrichr(
            gene_list=sig_genes,
            gene_sets=['GO_Biological_Process_2023', 'Reactome_2024', 'WikiPathways_2024_Mouse'],        
            outdir=None,
            verbose=False
        )
        
        # Store results
        res_df = enr.results
        sig_res = res_df[res_df["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
        enrichment_results[ct] = sig_res

        print(f"--- Top 3 Pathways for {ct} ---")
        print(sig_res[["Term", "Adjusted P-value", "Combined Score"]].head(3))
        print("\n")

        # takes 3 min

In [ ]:
if run:    

    target_ct = "Late Mature"
    df = enrichment_results[target_ct].copy()

    df = df.sort_values("Adjusted P-value").head(10)
    if df.empty:
        raise ValueError(f"No enriched terms to plot for {target_ct}")

    df = df.iloc[::-1]  # reverse for horizontal bar plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        df["Term"],
        -np.log10(df["Adjusted P-value"]),
        color="steelblue"
    )
    ax.set_xlabel("-log10(Adjusted P-value)")
    ax.set_title(f"Top enriched pathways: {target_ct}")
    plt.tight_layout()
    plt.show()

In [ ]:
if run:
    print(list(enrichment_results.keys()))

Conserved marker analysis, do it on single cell level if a package exists. We will try it using scanpy and then find an intersect

In [ ]:
# conserved markers for WT and Test
adata_WT = adata[adata.obs["treatment"]=="WT"] # keep
adata_Test = adata[adata.obs["treatment"]=="Test"] # change to different Tests of different experiments

In [ ]:
if run:    
    sc.tl.rank_genes_groups(
        adata_WT,
        groupby="celltype_final_2026",
        method="wilcoxon"
    )

    sc.tl.rank_genes_groups(
        adata_Test,
        groupby="celltype_final_2026",
        method="wilcoxon"
    )

In [ ]:
if run:
    wt_markers = sc.get.rank_genes_groups_df(
        adata_WT,
        group="Pre_cDC1"
    )

    test_markers = sc.get.rank_genes_groups_df(
        adata_Test,
        group="Pre_cDC1"
    )

In [ ]:
adata.obs["celltype_final_2026"].unique()

In [ ]:
if run:
    conserved = wt_markers.merge(
        test_markers,
        on="names",
        suffixes=("_WT", "_Test")
    )

    # filtering to remove all non-interesting genes
    conserved = conserved[
        (conserved["pvals_adj_WT"] < 0.05) &
        (conserved["pvals_adj_Test"] < 0.05) &
        (conserved["logfoldchanges_WT"] > 1) &
        (conserved["logfoldchanges_Test"] > 1)
    ]

    conserved

In [ ]:
if run:
    conserved["combined_padj"] = (
        conserved["pvals_adj_WT"] *
        conserved["pvals_adj_Test"]
    )

    conserved = conserved.sort_values(
        "combined_padj", # can sort on something different too, or set ascending = True or False then (not needed for pval)
    )

    conserved

    # scores are the test statistic to rank genes for DE. 
    # it represents how strongly a gene separates the selected group from the reference group
    # a positive score means the gene is enriched in the target cell type, a negative score means the gene is higher in the reference group

Conserved marker analysis for each celltype apart, to see which markers are conserved, meaning they satisfy a certain condition across all experiments for a particular celltype (they are significantly upregulated across all experiments)

In [ ]:
if True:
    # because Toxo contains WT and Test, split it into separate groups
    adata.obs["marker_group"] = adata.obs["experiment"].astype(str)

    adata.obs.loc[
        (adata.obs["experiment"] == "CITEseq_Toxo") &
        (adata.obs["treatment"] == "WT"),
        "marker_group"
    ] = "CITEseq_Toxo_WT"

    adata.obs.loc[
        (adata.obs["experiment"] == "CITEseq_Toxo") &
        (adata.obs["treatment"] == "Test"),
        "marker_group"
    ] = "CITEseq_Toxo_Test"

In [ ]:
if True:
    adata.obs.groupby("marker_group")["orig.ident"].unique()

In [ ]:
if True:
    # Store marker results
    # Structure:
    # marker_results[group][celltype] = marker dataframe

    marker_results = {}

    for group in adata.obs["marker_group"].unique():

        adata_group = adata[
            adata.obs["marker_group"] == group
        ].copy()

        # check upregulation or downregulation of all genes, using this rank test
        # each gene gets assigned a score (wilcoxon test score), logFC and pval
        sc.tl.rank_genes_groups(
            adata_group,
            groupby="celltype_final_2026",
            method="wilcoxon"
        )

        marker_results[group] = {}

        for ct in adata_group.obs["celltype_final_2026"].unique():

            markers = sc.get.rank_genes_groups_df(
                adata_group,
                group=ct
            )

            marker_results[group][ct] = markers


In [ ]:
if True:
    marker_results["CITEseq_Toxo_Test"]["Late Immature"]

In [ ]:
if True: 
    conserved_markers = {}

    celltypes = adata.obs["celltype_final_2026"].unique()

    for ct in celltypes:

        marker_tables = []
        n_experiments_present = 0

        for group, group_dict in marker_results.items():

            # Skip if celltype is absent in this experimental group
            if ct not in group_dict:
                continue

            n_experiments_present += 1

            markers = group_dict[ct].copy()

            # keep only significantly upregulated markers
            markers = markers[
                (markers["pvals_adj"] < 0.05) &
                (markers["logfoldchanges"] > 1)
            ]

            if markers.empty:
                continue

            marker_tables.append(
                markers[
                    [
                        "names",
                        "logfoldchanges",
                        "pvals_adj"
                    ]
                ].rename(
                    columns={
                        "logfoldchanges": f"logFC_{group}",
                        "pvals_adj": f"padj_{group}"
                    }
                )
            )


        if len(marker_tables) == 0:
            continue

        conserved = marker_tables[0]

        # this is where conserved markers are found, meaning they are merged across WT and Test
        # significance and upregulation was filtered before 
        for table in marker_tables[1:]:
            conserved = conserved.merge(
                table,
                on="names"
            )


        logfc_cols = [
            c for c in conserved.columns
            if c.startswith("logFC_")
        ]

        padj_cols = [
            c for c in conserved.columns
            if c.startswith("padj_")
        ]


        conserved["mean_logFC"] = (
            conserved[logfc_cols]
            .mean(axis=1)
        )

        conserved["highest_padj"] = (
            conserved[padj_cols]
            .max(axis=1)
        )

        conserved["combined_padj"] = (
            conserved[padj_cols]
            .prod(axis=1)
        )

        # Add metadata about conservation strength
        conserved["n_experiments_present"] = n_experiments_present

        conserved_markers[ct] = conserved.sort_values(
            "combined_padj"
        )

In [ ]:
if True: 
        celltypes

In [ ]:
# plot the table
if True:
    conserved_markers["Late Mature"]

In [ ]:
if True:
    # as a check to see if the table before is right, check which celltypes are missing from which group
    # All cell types present in the complete dataset
    all_celltypes = set(adata.obs["celltype_final_2026"].unique())

    rows = []

    for group in sorted(adata.obs["marker_group"].unique()):

        present = set(
            adata.obs.loc[
                adata.obs["marker_group"] == group,
                "celltype_final_2026"
            ].unique()
        )

        missing = sorted(all_celltypes - present)

        rows.append({
            "marker_group": group,
            "missing_celltypes": ", ".join(missing) if missing else "None"
        })

    missing_table = pd.DataFrame(rows)

    missing_table

Export the conserved marker results

In [ ]:
if run:    
    # for the pdf later
    for ct, df in conserved_markers.items():

        n_exp = sum(
            ct in group_dict
            for group_dict in marker_results.values()
        )

        df["n_experiments_present"] = n_exp
        conserved_markers[ct] = df

In [ ]:
if run:
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib.pagesizes import landscape, A4


    output_pdf = "conserved_markers_summary.pdf"

    doc = SimpleDocTemplate(
        output_pdf,
        pagesize=landscape(A4)
    )

    styles = getSampleStyleSheet()
    elements = []

    # for all celltypes
    for celltype, df in conserved_markers.items():

        # Skip empty results
        if df.empty:
            continue

        df = df.copy()

        # Identify experiment-specific columns
        logfc_cols = [
            c for c in df.columns
            if c.startswith("logFC_")
        ]

        padj_cols = [
            c for c in df.columns
            if c.startswith("padj_")
        ]

        # Add summary statistics if not already present
        df["mean_logFC"] = (
            df[logfc_cols]
            .mean(axis=1)
        )

        df["highest_padj"] = (
            df[padj_cols]
            .max(axis=1)
        )

        # Sort strongest conserved markers first
        summary = (
            df[
                [
                    "names",
                    "mean_logFC",
                    "highest_padj",
                    "combined_padj"
                ]
            ]
            .sort_values(
                "mean_logFC",
                ascending=False
            )
            .head(50)
        )

        # retrieve number of experiments where this celltype was present
        n_exp = df["n_experiments_present"].iloc[0]


        # Header
        elements.append(
            Paragraph(
                f"Conserved markers: {celltype}",
                styles["Heading2"]
            )
        )

        elements.append(
            Paragraph(
                f"""
                Present in {n_exp}/{len(marker_results)} experimental groups<br/>
                Number of conserved markers: {df['names'].nunique()}""",
                styles["Normal"]
            )
        )

        elements.append(
            Spacer(1, 10)
        )


        # Table
        table_data = [
            [
                "Gene",
                "Mean log2FC",
                "Highest adjusted p-value",
                "Combined adjusted p-value"
            ]
        ]

        for _, row in summary.iterrows():

            table_data.append(
                [
                    row["names"],
                    round(row["mean_logFC"], 3),
                    f"{row['highest_padj']:.2e}",
                    f"{row['combined_padj']:.2e}"
                ]
            )


        table = Table(
            table_data,
            repeatRows=1
        )

        table.setStyle(
            TableStyle(
                [
                    ("GRID", (0,0), (-1,-1), 0.25, None),
                    ("VALIGN", (0,0), (-1,-1), "TOP"),
                ]
            )
        )

        elements.append(table)

        # Start every cell type on a new page
        elements.append(PageBreak())


    doc.build(elements)

    print(f"Saved PDF: {output_pdf}")

Next, do the same but for all genes over all experiments, with a significance column at the end. 

The key difference with conserved markers, is that now we take all genes for each celltype and compare them across all experiments in which they appear. For the conserved genes, we filter all non-upregulated and non-significant genes out before merging Test and WT

In [ ]:
if run:    
    output_excel = "all_ranked_markers_summary.xlsx"

    all_celltypes = adata.obs["celltype_final_2026"].unique()

    with pd.ExcelWriter(output_excel) as writer:

        for ct in all_celltypes:

            experiment_tables = []

            for group, group_dict in marker_results.items():

                if ct not in group_dict:
                    continue

                markers = group_dict[ct].copy()

                markers = markers[
                    [
                        "names",
                        "logfoldchanges",
                        "pvals_adj",
                        "scores"
                    ]
                ]
                # rename cols
                markers = markers.rename(
                    columns={
                        "logfoldchanges": f"logFC_{group}",
                        "pvals_adj": f"padj_{group}",
                        "scores": f"score_{group}"
                    }
                )

                experiment_tables.append(markers)


            if len(experiment_tables) == 0:
                continue


            # Merge all experiments by gene
            merged = experiment_tables[0]

            for table in experiment_tables[1:]:
                merged = merged.merge(
                    table,
                    on="names",
                    how="outer"
                )


            # Summary statistics

            logfc_cols = [
                c for c in merged.columns
                if c.startswith("logFC_")
            ]

            padj_cols = [
                c for c in merged.columns
                if c.startswith("padj_")
            ]


            # Count how many experiments show significant marker behaviour
            merged["n_experiments_significant"] = (
                merged[padj_cols]
                .lt(0.05)
                .sum(axis=1)
            )


            # Average logFC over experiments where gene was detected
            merged["mean_logFC"] = (
                merged[logfc_cols]
                .mean(axis=1, skipna=True)
            )


            # Best significance across experiments
            merged["max_padj"] = (
                merged[padj_cols]
                .max(axis=1, skipna=True)
            )

            # min logFC
            merged["min_logFC"] = (
            merged[logfc_cols]
            .min(axis=1, skipna=True)
            )


            # Rank genes, according to these 2 metrics
            merged = merged.sort_values(
                [
                    "n_experiments_significant",
                    "mean_logFC"
                ],
                ascending=[
                    False,
                    False
                ]
            )


            # Write one sheet per cell type
            merged.to_excel(
                writer,
                sheet_name=ct[:31],
                index=False
            )


    print(f"Saved: {output_excel}")

To visualize it in this notebook:

In [ ]:
if run:    
    all_marker_tables = {}

    all_celltypes = adata.obs["celltype_final_2026"].unique()

    # total number of experimental groups
    n_total_experiments = len(marker_results)


    for ct in all_celltypes:

        experiment_tables = []

        # count in how many experiments this celltype exists
        n_experiments_present = 0

        for group, group_dict in marker_results.items():

            if ct not in group_dict:
                continue

            n_experiments_present += 1

            markers = group_dict[ct][
                [
                    "names",
                    "logfoldchanges",
                    "pvals_adj",
                    "scores"
                ]
            ].copy()

            markers = markers.rename(
                columns={
                    "logfoldchanges": f"logFC_{group}",
                    "pvals_adj": f"padj_{group}",
                    "scores": f"score_{group}"
                }
            )

            experiment_tables.append(markers)


        if len(experiment_tables) == 0:
            continue


        # Merge experiments by gene
        merged = experiment_tables[0]

        for table in experiment_tables[1:]:

            merged = merged.merge(
                table,
                on="names",
                how="outer"
            )


        logfc_cols = [
            c for c in merged.columns
            if c.startswith("logFC_")
        ]

        padj_cols = [
            c for c in merged.columns
            if c.startswith("padj_")
        ]


        # How often is this gene significant?
        merged["n_experiments_significant"] = (
            merged[padj_cols]
            .lt(0.05)
            .sum(axis=1)
        )


        # Cell type-level conservation
        merged["n_experiments_present"] = n_experiments_present

        merged["n_total_experiments"] = n_total_experiments

        merged["celltype_presence_fraction"] = (
            n_experiments_present / n_total_experiments
        )


        # Gene-level summary
        merged["mean_logFC"] = (
            merged[logfc_cols]
            .mean(axis=1, skipna=True)
        )

        merged["max_padj"] = (
            merged[padj_cols]
            .max(axis=1, skipna=True)
        )


        merged = merged.sort_values(
            [
                "n_experiments_significant",
                "mean_logFC"
            ],
            ascending=[
                False,
                False
            ]
        )


        all_marker_tables[ct] = merged

Now, to create a filtering mechanism that returns genes satisfying certain conditions (pval adj, min logFC, n_experiments) per celltype

In [ ]:
# this is the function that  accepts certain cutoffs, which you specify
def find_markers_from_excel(
    celltype,
    max_padj=0.05,
    mean_logFC=1,
    n_experiments_significant=3
):
    
    # Load only the requested celltype sheet
    df = pd.read_excel(
        "c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Downstream_processing/all_ranked_markers_summary.xlsx", # hardcoded, can maybe change this 
        sheet_name=celltype
    )

    # Apply filtering criteria
    filtered = df[
        (df["max_padj"] <= max_padj) &
        (df["mean_logFC"] >= mean_logFC) &
        (df["n_experiments_significant"] >= n_experiments_significant)
    ].copy()


    # Sort strongest markers first
    filtered = filtered.sort_values(
        [
            "n_experiments_significant",
            "mean_logFC",
            "max_padj"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )


    print(
        f"{len(filtered)} genes found for {celltype}"
    )

    return filtered

In [ ]:
# test the function
# specify the following in this order: celltype, max adjusted p-value, mean log fold change, amount of experiments in which the gene is significant
find_markers_from_excel("Proliferating_cDC1", 0.005, 2, 10)

GRN analysis (Decoupler), do it on bulk.

We use the following approach: we take the DESeq2 data to compare differences in TF levels between test and WT, resulting in differential TF activity between the 2 selected groups. For toxo, this is against its own WT group, for LNP, this is against the LNP_WT experiment. 

Another approach is just TF inference in different groups, without the comparison (DE). The ULM method takes a matrix of genes x samples/profiles and a TF target network (collectri), and then fits for each TF, a linear model relating the observed gene level values to the TF's target weights. So the input can be a expression, normalized expression or even DESeq2 gene level statistics. Using DESeq2 data results iin a differential TF activity analysis, while using counts just identifies the TFs.

In [ ]:
pdata

In [ ]:
collectri = dc.op.collectri(organism="mouse")
collectri

In [ ]:
results_toxo_final

In [ ]:
tf_scores_toxo = {}
tf_padj_toxo = {}

for exp, ct in results_toxo_final.keys():

    print(f"\n================ {ct} ================")

    results = results_toxo_final[(exp, ct)].copy()

    if "stat" not in results.columns:
        print(f"Skipping {ct}: no 'stat' column")
        continue

    results = (
        results
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["stat"])
    )

    if results.empty:
        print(f"Skipping {ct}: no valid DESeq2 statistics")
        continue

    # Genes = columns
    # One row = Test vs WT for this celltype
    network_data = (
        results[["stat"]]
        .T
        .rename(index={"stat": "Test_vs_WT"})
        .astype(float)
    )

    # Run ULM
    tf_score_toxo_DE, tf_pval_toxo_DE = dc.mt.ulm(
        data=network_data,
        net=collectri
    )

    # Store separately for this celltype
    tf_scores_toxo[(exp, ct)] = tf_score_toxo_DE
    tf_padj_toxo[(exp, ct)] = tf_pval_toxo_DE

In [ ]:
tf_scores_toxo

In [ ]:
# plot results in a heatmap for each celltype 
# because we did DESeq2 before, there is no meaningful Test and WT groups, only the DESeq2 data
# the scale are TF scores from ULM

for (exp, ct), scores in tf_scores_toxo.items():

    # scores = 1 row (Test_vs_WT) × TFs

    # Select top 50 TFs by absolute ULM score
    top_tfs = (
        scores
        .T
        .iloc[:, 0]
        .abs()
        .sort_values(ascending=False)
        .head(50)
        .index
    )

    heatmap_data = scores.loc[:, top_tfs].T
    heatmap_data.columns = ["Test vs WT"]

    # Plot one heatmap for this celltype
    sns.clustermap(
        heatmap_data,
        cmap="vlag",
        center=0,
        figsize=(20, 20),
        linewidths=0.2,
        col_cluster=False,
        row_cluster=True
    )

    plt.suptitle(
        f"TF activity from DESeq2 — {ct}",
        y=1.02
    )

    plt.show()

In [ ]:
    # pdf
if run:
    pdf_path = "Toxo_DESeq2_ULM_TF_activity.pdf"

    with PdfPages(pdf_path) as pdf:

        # plot results in a heatmap for each celltype
        #
        # because we did DESeq2 before, there is no meaningful Test and WT
        # groups, only the DESeq2 data

        for (exp, ct), scores in tf_scores_toxo.items():

            # scores = 1 row (Test_vs_WT) × TFs

            # Select top 50 TFs by absolute ULM score
            top_tfs = (
                scores
                .T
                .iloc[:, 0]
                .abs()
                .sort_values(ascending=False)
                .head(50)
                .index
            )

            heatmap_data = scores.loc[:, top_tfs].T
            heatmap_data.columns = ["Test vs WT"]

            # Plot one heatmap for this celltype
            g = sns.clustermap(
                heatmap_data,
                cmap="vlag",
                center=0,
                figsize=(20, 20),
                linewidths=0.2,
                col_cluster=False,
                row_cluster=True
            )

            g.fig.suptitle(
                f"TF activity from DESeq2 — {ct}",
                y=1.02
            )

            # Save this celltype's heatmap as one PDF page
            pdf.savefig(g.fig, bbox_inches="tight")

            plt.close(g.fig)

    print(f"Saved PDF: {pdf_path}")

Below is the TF inference, no DESeq2 data. This results in TF inference for each pseudobulk group individually. Here, we check the TF signature of each experimental test group

In [ ]:
# run ULM
# this function accepts the collectri map, which is a pre-determined TF map with target genes
# ULM does not discover TF gene relationships, it rather asks if the expression pattern of known targets of a TF look consistent with TF being active
# so it does not find new relations, it just checks which ones exist in this dataset
# I think SCENIC does find new relationships
dc.mt.ulm(
    data=pdata,
    net=collectri
)

tf_scores = pdata.obsm["score_ulm"]
tf_padj = pdata.obsm["padj_ulm"]
tf_scores

In [ ]:
pdata

In [ ]:
# Add metadata
# split on treatment, and celltype, fix
# make a heatmap per celltype
# make a dataframe per celltype
tf_scores_meta = tf_scores.copy()

tf_scores_meta["experiment"] = (
    pdata.obs["experiment"].astype(str).values
)

tf_scores_meta["treatment"] = (
    pdata.obs["treatment"].astype(str).values
)

tf_scores_meta["celltype"] = (
    pdata.obs["celltype_final_2026"].astype(str).values
)


# Create unified grouping as normal strings
tf_scores_meta["GRN_group"] = (
    tf_scores_meta["experiment"].astype(str)
)

# Toxo WT -> common WT group
tf_scores_meta.loc[
    (tf_scores_meta["experiment"] == "CITEseq_Toxo") &
    (tf_scores_meta["treatment"] == "WT"),
    "GRN_group"
] = "CITEseq_Toxo_WT"

# Toxo Test -> separate Test group
tf_scores_meta.loc[
    (tf_scores_meta["experiment"] == "CITEseq_Toxo") &
    (tf_scores_meta["treatment"] == "Test"),
    "GRN_group"
] = "CITEseq_Toxo_Test"

tf_scores_meta["GRN_group"].value_counts()

In [ ]:
# Make separate TF activity dataframe for each cell type

tf_by_celltype = {}

celltypes = tf_scores_meta["celltype"].unique()

for ct in celltypes:

    # Select this cell type
    ct_data = tf_scores_meta[
        tf_scores_meta["celltype"] == ct
    ].copy()

    # Keep only TF score columns
    tf_columns = tf_scores.columns

    # Add GRN group to the TF scores
    ct_scores = ct_data[["GRN_group"] + list(tf_columns)]

    # Average TF activity within each experimental group
    ct_mean = (
        ct_scores
        .groupby("GRN_group")[list(tf_columns)]
        .mean()
    )

    # Store
    tf_by_celltype[ct] = ct_mean

In [ ]:
tf_by_celltype # so per celtype as key it holds all experiments, and with those, the TF activity scores (not logFC)
# before, we had activity scores per TF in a celltype in a specific mouse (pseudobulk group)
# now, we have averaged the activity score of each TF in an experimental group for a specific celltype

In [ ]:
# find out which TFs are conserved across the experimental groups, meaning they are active, with similar magnitude in the same direction (positive or negative across all experimental groups)
def find_conserved_tfs(
    tf_dict,
    min_activity=1.0, # minimal activity score
    min_fraction=0.8 # fraction of experiments with activity above threshold (the min_activity)
):
    conserved = {}

    for celltype, df in tf_dict.items():

        results = []

        for tf in df.columns:

            scores = df[tf].dropna()

            if len(scores) == 0:
                continue

            # Fraction of experiments with activity above threshold
            positive_fraction = (scores >= min_activity).mean()

            # Fraction with activity below negative threshold
            negative_fraction = (scores <= -min_activity).mean()

            # Determine whether consistently active or inactive
            if positive_fraction >= min_fraction:
                direction = "active"
                consistency = positive_fraction

            elif negative_fraction >= min_fraction:
                direction = "inactive"
                consistency = negative_fraction

            else:
                continue

            results.append({
                "TF": tf,
                "direction": direction,
                "mean_activity": scores.mean(),
                "mean_abs_activity": scores.abs().mean(),
                "min_activity": scores.min(),
                "max_activity": scores.max(),
                "consistency": consistency
            })

        if results:
            conserved[celltype] = (
                pd.DataFrame(results)
                .sort_values(
                    ["direction", "mean_abs_activity"],
                    ascending=[True, False]
                )
            )

    return conserved

In [ ]:
conserved_tfs = find_conserved_tfs(tf_by_celltype)

Here, we plot a heatmap per celltype, and plot the top TFs across the experimental groups on their z-score. GOAL: to see which TFs are more/less active in which celltypes in which treatments. 

The z-score (standard score) tells how many standard deviations a data point is above or below the mean of the group. A positive score means it is above the mean, negative means it is below the mean. Note, a z-score is a statistic which measures the difference from the mean, whatever that mean may be. It does not say anything about the TF having positive or negative expression. 

In [ ]:
# this plotting DOES NOT make a heatmap for "Other cDC1s", because these are only present in 1 experimental group, making the z-score useless because
# there is no comparison possible

for ct, tf_heatmap in tf_by_celltype.items():

    # Need at least 2 experimental groups
    if tf_heatmap.shape[0] < 2:
        print(
            f"Skipping {ct}: only "
            f"{tf_heatmap.shape[0]} experimental group(s)"
        )
        continue

    # Select most variable TFs
    top_tfs = (
        tf_heatmap.var(axis=0)
        .sort_values(ascending=False)
        .head(50)
        .index
    )

    tf_heatmap = tf_heatmap[top_tfs]

    # Remove TFs with missing/non-finite values
    tf_heatmap = tf_heatmap.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna(axis=1)

    # Need at least one TF
    if tf_heatmap.shape[1] == 0:
        print(
            f"Skipping {ct}: no usable TFs"
        )
        continue

    # Z-score across experimental groups
    tf_heatmap_z = tf_heatmap.apply(
        zscore,
        axis=0
    )

    # Remove TFs that produced NaN
    tf_heatmap_z = tf_heatmap_z.dropna(axis=1)

    if tf_heatmap_z.shape[1] == 0:
        print(
            f"Skipping {ct}: no TFs with valid variance"
        )
        continue

    sns.clustermap(
        tf_heatmap_z,
        cmap="vlag",
        center=0,
        figsize=(20, 4),
        linewidths=0.2
    )

    plt.suptitle(
        f"TF activity — {ct}",
        y=1.02
    )

    plt.show()

    # scale is z-score, not logFC, not activity score

In [ ]:
# instead of heatmaps, write it out to excel

if run: 

    output_file = "TF_activity_zscores_by_celltype_pdata.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for ct, tf_heatmap in tf_by_celltype.items():

            # Need at least 2 experimental groups
            if tf_heatmap.shape[0] < 2:
                print(
                    f"Skipping {ct}: only "
                    f"{tf_heatmap.shape[0]} experimental group(s)"
                )
                continue

            # Remove non-finite values
            tf_heatmap = tf_heatmap.replace(
                [np.inf, -np.inf],
                np.nan
            ).dropna(axis=1)

            # Need at least one TF
            if tf_heatmap.shape[1] == 0:
                print(
                    f"Skipping {ct}: no usable TFs"
                )
                continue

            # Z-score each TF across experimental groups
            tf_heatmap_z = tf_heatmap.apply(
                zscore,
                axis=0
            )

            # Remove TFs with zero variance / NaN z-scores
            tf_heatmap_z = tf_heatmap_z.dropna(axis=1)

            if tf_heatmap_z.shape[1] == 0:
                print(
                    f"Skipping {ct}: no TFs with valid variance"
                )
                continue

            # TFs = rows
            # Experimental groups = columns
            tf_heatmap_z = tf_heatmap_z.T

            # Write one sheet per celltype
            # Excel sheet names cannot exceed 31 characters
            sheet_name = str(ct)[:31]

            tf_heatmap_z.to_excel(
                writer,
                sheet_name=sheet_name
            )

            print(
                f"Wrote {ct}: "
                f"{tf_heatmap_z.shape[0]} TFs × "
                f"{tf_heatmap_z.shape[1]} experimental groups"
            )

    print(f"\nSaved to: {output_file}")

Make pdf from this

In [ ]:
if run:    

    pdf_path = "TF_activity_zscores_by_celltype_pdata.pdf"

    with PdfPages(pdf_path) as pdf:

        for ct, tf_heatmap in tf_by_celltype.items():

            # ---------------------------------------------
            # Keep only numeric TF activity columns
            # ---------------------------------------------
            tf_heatmap = tf_heatmap.select_dtypes(include=np.number)

            # Remove non-finite values
            tf_heatmap = tf_heatmap.replace(
                [np.inf, -np.inf],
                np.nan
            )

            # Select most variable TFs
            top_tfs = (
                tf_heatmap.var(axis=0)
                .sort_values(ascending=False)
                .head(50)
                .index
            )

            tf_heatmap = tf_heatmap[top_tfs]

            # Remove TFs containing missing values
            tf_heatmap = tf_heatmap.dropna(axis=1)

            if tf_heatmap.shape[0] < 2:
                print(f"Skipping {ct}: fewer than 2 experimental groups")
                continue

            if tf_heatmap.shape[1] < 2:
                print(f"Skipping {ct}: fewer than 2 TFs")
                continue

            # ---------------------------------------------
            # Z-score each TF across experimental groups
            # ---------------------------------------------
            tf_heatmap_z = tf_heatmap.apply(
                lambda x: (
                    (x - x.mean()) / x.std(ddof=0)
                    if x.std(ddof=0) != 0
                    else np.nan
                ),
                axis=0
            )

            # Remove TFs that could not be z-scored
            tf_heatmap_z = tf_heatmap_z.dropna(axis=1)

            if tf_heatmap_z.shape[1] < 2:
                print(f"Skipping {ct}: insufficient variable TFs")
                continue

            # ---------------------------------------------
            # Clustered heatmap
            #
            # Rows    = TFs
            # Columns = experimental groups
            # ---------------------------------------------
            g = sns.clustermap(
                tf_heatmap_z,
                cmap="vlag",
                center=0,
                figsize=(12, 10),
                linewidths=0.2,
                linecolor="lightgray",
                xticklabels=True,
                yticklabels=True,
                method="average",
                metric="euclidean",
                row_cluster=True,
                col_cluster=True,
                cbar_kws={
                    "label": "TF activity z-score"
                }
            )

            # ---------------------------------------------
            # Title
            # ---------------------------------------------
            g.fig.suptitle(
                f"TF activity across experimental groups\n{ct}",
                fontsize=16,
                fontweight="bold",
                y=1.02
            )

            # ---------------------------------------------
            # Axis labels
            # ---------------------------------------------
            g.ax_heatmap.set_xlabel(
                "Experimental group"
            )

            g.ax_heatmap.set_ylabel(
                "TF"
            )

            # Rotate experimental-group labels
            plt.setp(
                g.ax_heatmap.get_xticklabels(),
                rotation=45,
                ha="right"
            )

            # Keep TF labels horizontal
            plt.setp(
                g.ax_heatmap.get_yticklabels(),
                rotation=0
            )

            # ---------------------------------------------
            # Save to PDF
            # ---------------------------------------------
            pdf.savefig(
                g.fig,
                bbox_inches="tight"
            )

            plt.close(g.fig)

            print(
                f"Added {ct} "
                f"({len(tf_heatmap_z.columns)} TFs)"
            )

    print(f"\nPDF saved as: {pdf_path}")

Finally, we test which TFs differ between the Toxo and the other immunogenic groups. Decoupler has a network plotting for this. 

We are interested in Toxo vs the other immunogenic experiments

In [ ]:
# current tf_scores already holds scores on the pseudobulk data
tf_scores

# this holds the TF scores, not based on DESeq2 data

In [ ]:
other_groups = [
    "CITEseq_LNP_CpG_LNPs",
    "CITEseq_LNP_eLNPs",
    "CITEseq_LNP_pIC_LNPs",
    "CITEseq_LNP_pIC"
]

all_groups = [
    "CITEseq_Toxo_Test",
    "CITEseq_LNP_CpG_LNPs",
    "CITEseq_LNP_eLNPs",
    "CITEseq_LNP_pIC_LNPs",
    "CITEseq_LNP_pIC"
]


# Choose celltype
celltype = "Late Mature"

# TF activity for this celltype
activity = tf_by_celltype[celltype]

# --------------------------------------------------
# 1. Select TFs based on Toxo vs mean of other groups
# --------------------------------------------------

# find Toxo specific TFs by subtracting the mean of every TF from the other groups
comparison = (
    activity.loc["CITEseq_Toxo_Test"]
    - activity.loc[other_groups].mean(axis=0)
)

# sort on abs values to find them (abs so that you also find negative)
top_tfs = (
    comparison.abs()
    .sort_values(ascending=False)
    .head(50)
    .index
    .tolist()
)

# --------------------------------------------------
# 2. Now retrieve ORIGINAL activity scores
#    for those TFs in every individual group
# --------------------------------------------------

heatmap_data = activity.loc[
    all_groups,
    top_tfs
]

# --------------------------------------------------
# 3. Z-score each TF across the individual groups
# --------------------------------------------------

heatmap_z = heatmap_data.apply(
    lambda x: (x - x.mean()) / x.std(),
    axis=0
)

# --------------------------------------------------
# 4. Plot
# --------------------------------------------------

sns.clustermap(
    heatmap_z,
    cmap="vlag",
    center=0,
    figsize=(30, 8),
    annot=True,
    # fmt=".2f"
)

plt.suptitle(
    f"TF activity of Toxo-associated TFs — {celltype}",
    y=1.02
)

plt.show()

In [ ]:
# make a pdf from these heatmaps
if run:
    other_groups = [
        "CITEseq_LNP_CpG_LNPs",
        "CITEseq_LNP_eLNPs",
        "CITEseq_LNP_pIC_LNPs",
        "CITEseq_LNP_pIC"
    ]

    all_groups = [
        "CITEseq_Toxo_Test",
        "CITEseq_LNP_CpG_LNPs",
        "CITEseq_LNP_eLNPs",
        "CITEseq_LNP_pIC_LNPs",
        "CITEseq_LNP_pIC"
    ]

    output_file = "Toxo_associated_TF_activity_vs_immuno.pdf"

    with PdfPages(output_file) as pdf:

        for celltype, activity in tf_by_celltype.items():

            print(f"Processing: {celltype}")

            try:

                # --------------------------------------------------
                # Check that all required groups are present
                # --------------------------------------------------

                required_groups = set(all_groups)

                missing_groups = (
                    required_groups -
                    set(activity.index)
                )

                if missing_groups:
                    print(
                        f"  Skipping {celltype}: "
                        f"missing groups {sorted(missing_groups)}"
                    )
                    continue

                # --------------------------------------------------
                # 1. Select TFs based on Toxo vs mean of other groups
                # --------------------------------------------------

                comparison = (
                    activity.loc["CITEseq_Toxo_Test"]
                    - activity.loc[other_groups].mean(axis=0)
                )

                # Remove non-finite values
                comparison = comparison.replace(
                    [np.inf, -np.inf],
                    np.nan
                ).dropna()

                if comparison.empty:
                    print(
                        f"  Skipping {celltype}: "
                        f"no valid TF comparisons"
                    )
                    continue

                # Select top 50 Toxo-associated TFs
                top_tfs = (
                    comparison.abs()
                    .sort_values(ascending=False)
                    .head(50)
                    .index
                    .tolist()
                )

                # --------------------------------------------------
                # 2. Retrieve ORIGINAL activity scores
                # --------------------------------------------------

                heatmap_data = activity.loc[
                    all_groups,
                    top_tfs
                ].copy()

                # Remove TFs containing invalid values
                heatmap_data = heatmap_data.replace(
                    [np.inf, -np.inf],
                    np.nan
                ).dropna(axis=1)

                if heatmap_data.shape[1] == 0:
                    print(
                        f"  Skipping {celltype}: "
                        f"no TFs with complete activity scores"
                    )
                    continue

                # --------------------------------------------------
                # 3. Z-score each TF across the individual groups
                # --------------------------------------------------

                heatmap_z = heatmap_data.apply(
                    lambda x: (x - x.mean()) / x.std(),
                    axis=0
                )

                # Remove TFs for which z-score failed
                heatmap_z = heatmap_z.replace(
                    [np.inf, -np.inf],
                    np.nan
                ).dropna(axis=1)

                if heatmap_z.shape[1] == 0:
                    print(
                        f"  Skipping {celltype}: "
                        f"no TFs with valid z-scores"
                    )
                    continue

                # --------------------------------------------------
                # 4. Plot
                # --------------------------------------------------

                g = sns.clustermap(
                    heatmap_z,
                    cmap="vlag",
                    center=0,
                    figsize=(30, 8),
                    annot=True,
                    fmt=".2f",
                    linewidths=0.2
                )

                g.fig.suptitle(
                    f"TF activity of Toxo-associated TFs — {celltype}",
                    y=1.02
                )

                # Save this celltype as one PDF page
                pdf.savefig(g.fig, bbox_inches="tight")
                plt.close(g.fig)

                print(
                    f"  Added to PDF: "
                    f"{heatmap_z.shape[1]} TFs"
                )

            except Exception as e:

                print(
                    f"  ERROR for {celltype}: "
                    f"{type(e).__name__}: {e}"
                )

                # Make absolutely sure a failed plot does not
                # contaminate the next iteration
                plt.close("all")

    print(f"\nPDF saved to: {output_file}")

In [ ]:
    # same thing, but now excel
if run: 
    other_groups = [
        "CITEseq_LNP_CpG_LNPs",
        "CITEseq_LNP_eLNPs",
        "CITEseq_LNP_pIC_LNPs",
        "CITEseq_LNP_pIC"
    ]

    all_groups = [
        "CITEseq_Toxo_Test",
        "CITEseq_LNP_CpG_LNPs",
        "CITEseq_LNP_eLNPs",
        "CITEseq_LNP_pIC_LNPs",
        "CITEseq_LNP_pIC"
    ]

    output_file = "Toxo_associated_TF_activity_vs_immuno.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for celltype, activity in tf_by_celltype.items():

            print(f"Processing: {celltype}")

            try:

                # --------------------------------------------------
                # Check that all required groups are present
                # --------------------------------------------------

                missing_groups = (
                    set(all_groups) - set(activity.index)
                )

                if missing_groups:
                    print(
                        f"  Skipping {celltype}: "
                        f"missing groups {sorted(missing_groups)}"
                    )
                    continue

                # --------------------------------------------------
                # 1. Select TFs based on Toxo vs mean of other groups
                #
                # Here we KEEP ALL TFs
                # --------------------------------------------------

                comparison = (
                    activity.loc["CITEseq_Toxo_Test"]
                    - activity.loc[other_groups].mean(axis=0)
                )

                # --------------------------------------------------
                # 2. Retrieve ORIGINAL activity scores
                # --------------------------------------------------

                heatmap_data = activity.loc[
                    all_groups,
                    comparison.index
                ].copy()

                # Remove non-finite values
                heatmap_data = heatmap_data.replace(
                    [np.inf, -np.inf],
                    np.nan
                )

                # Remove TFs with missing values
                heatmap_data = heatmap_data.dropna(axis=1)

                if heatmap_data.shape[1] == 0:
                    print(
                        f"  Skipping {celltype}: "
                        f"no TFs with complete activity scores"
                    )
                    continue

                # --------------------------------------------------
                # 3. Z-score each TF across the five groups
                # --------------------------------------------------

                tf_z = heatmap_data.apply(
                    lambda x: (x - x.mean()) / x.std(),
                    axis=0
                )

                # Remove TFs with invalid z-scores
                tf_z = tf_z.replace(
                    [np.inf, -np.inf],
                    np.nan
                ).dropna(axis=1)

                if tf_z.shape[1] == 0:
                    print(
                        f"  Skipping {celltype}: "
                        f"no TFs with valid z-scores"
                    )
                    continue

                # --------------------------------------------------
                # 4. TFs as rows, experimental groups as columns
                # --------------------------------------------------

                tf_z = tf_z.T

                # --------------------------------------------------
                # 5. Excel sheet
                # --------------------------------------------------

                sheet_name = str(celltype)[:31]

                tf_z.to_excel(
                    writer,
                    sheet_name=sheet_name
                )

                print(
                    f"  Wrote {tf_z.shape[0]} TFs × "
                    f"{tf_z.shape[1]} experimental groups"
                )

            except Exception as e:

                print(
                    f"  ERROR for {celltype}: "
                    f"{type(e).__name__}: {e}"
                )

    print(f"\nExcel file saved to: {output_file}")

In [ ]:
comparison.abs().sort_values(ascending=False)

Now we try to find which genes in our dataset drive the inference of the TFs we found using previous analyses, especially in the specific immunogenic programs. Using DESeq2 results

In [ ]:
results_toxo_vs_other # contains a dict for each celltype, and experiment associated
# this compared toxo test vs test of one specific immunogenic experiment
# nt, for each celltype, resulting in 4 different comparisons actually

In [ ]:
# work input from DESeq2, now Late_Mature
immunogenic_list = [
    "CITEseq_LNP_CpG_LNPs",
    "CITEseq_LNP_eLNPs",
    "CITEseq_LNP_pIC",
    "CITEseq_LNP_pIC_LNPs"
]
# Here, we speciify a celltype in a specific group
# the results_toxo_vs_other contains results from the DESeq2 between toxo and other immunogenic experimental groups (pseudobulk groups)
res = results_toxo_vs_other[("CITEseq_LNP_CpG_LNPs", "Late Mature")]

# take DESeq2 statistics, transpose the dataset (so genes become in the columns)
network_data_toxo_vs_immuno = (
    res[["stat"]]
    .T
    .rename(index={"stat": "Toxo_vs_CITEseq_LNP_CpG_LNPs"})
)

In [ ]:
network_data_toxo_vs_immuno

In [ ]:
# DESeq2 resulted in some NaN, which will give errors in the block after
network_data_toxo_vs_immuno = network_data_toxo_vs_immuno.replace(
    [np.inf, -np.inf],
    np.nan
).dropna(axis=1)

In [ ]:
# calculate which TFs are present based on the genes in the dataset
tf_scores, tf_padj = dc.mt.ulm(
    data=network_data_toxo_vs_immuno,
    net=collectri
)

In [ ]:
tf_scores

In [ ]:
tf_scores.T.sort_values("Toxo_vs_CITEseq_LNP_CpG_LNPs",ascending=False) # to visualize, which TFs score the highest in this group

In [ ]:
# get TFs from your inferred TF scores
top_tfs_toxo_vs_immuno = (
    tf_scores
    .T
    .sort_values("Toxo_vs_CITEseq_LNP_CpG_LNPs", ascending=False)
    .head(50)
    .index
    .tolist()
)

top_tfs_toxo_vs_immuno

In [ ]:
# Now that we have the top TFs for this comparison (toxo vs one immunogenic group), we want to know which genes contribute most to these TFs
top_tfs = top_tfs_toxo_vs_immuno

# only use target nodes that are present in our data too
network_top = collectri[
    collectri["source"].isin(top_tfs)
].copy()

network_top.head(20)

In [ ]:
# only look in the genes present in the data to find target genes
genes_in_data = set(pdata.var_names)

network_top = network_top[
    network_top["target"].isin(genes_in_data)
].copy()

In [ ]:
# use DESeq2 data
res = results_toxo_vs_other[
    ("CITEseq_LNP_CpG_LNPs", "Late Mature")
].copy()

res["gene"] = res.index

In [ ]:
# merge the network with the DE results
res["gene"] = res.index

network_de = network_top.merge(
    res[
        [
            "gene",
            "stat",
            "log2FoldChange",
            "padj"
        ]
    ],
    left_on="target",
    right_on="gene",
    how="inner"
)

In [ ]:
network_de # from this, you can say that these genes are candidate targets contributing to the inferred activity for the TF

In [ ]:
# calculate contribution score, calculated based on the weight of the target gene (how strongly does the TF regulate this) and the DE statistic 
# (how strongly the gene changes in the Toxo vs other comparison)
network_de["contribution"] = (
    network_de["weight"] *
    network_de["stat"]
)

In [ ]:
network_de

In [ ]:
network_de["contribution"] = (
    network_de["weight"] * network_de["stat"]
)

network_de["abs_contribution"] = (
    network_de["contribution"].abs()
)

candidate_genes = (
    network_de
    .groupby("source", group_keys=False)
    .apply(
        lambda x: x.nlargest(10, "abs_contribution"),
        include_groups=False
    )
)

In [ ]:
candidate_genes

Next, we will identify modules based on the DESeq2 data. DESeq2 spits out a gene-level differential expression table between the groups we specified. This is a starting point for the module analysis, where we will identify genes in groups that could represent a program known or not known in databases. 

This can be done on several levels:
- co-expression
- pathway analysis (GSEA) 
- TF inference/ regulons (see before)

First, we will identify gene modules based on the conserved markers identified earlier, for each celltype individually. These lists will be used in another script "pyUCell.ipynb" to test

In [ ]:
celltypes

In [ ]:
# without print, it would be a vertical list, which is not good for copying
print(conserved_markers["Pre_cDC1"]["names"].to_list())  # None
print(conserved_markers["Proliferating_cDC1"]["names"].to_list()) 
print(conserved_markers["Early Immature"]["names"].to_list()) # None
print(conserved_markers["Late Immature"]["names"].to_list())
print(conserved_markers["Early Mature"]["names"].to_list())
print(conserved_markers["Late Mature"]["names"].to_list())
print(conserved_markers["Other cDC1s"]["names"].to_list()) 
print(conserved_markers["cDC1_engulfing_RBC"]["names"].to_list())

Now, we will try to find genes in the markers excel file which are conserved for a specific experiment across all celltypes

In [ ]:
# find significant genes in the dataset
def genes_significant_all_celltypes(
    excel_file,
    experiment,
    metric="padj",
    threshold=0.05
):
    xls = pd.ExcelFile(excel_file)
    genes_per_celltype = []

    col = f"{metric}_{experiment}"

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(excel_file, sheet_name=sheet)

            if col not in df.columns:
                raise ValueError(
                    f"Column '{col}' not found in sheet '{sheet}'"
                )

            # Convert comma decimal separator to dot
            values = pd.to_numeric(
                df[col].astype(str).str.replace(",", ".", regex=False),
                errors="coerce"
            )

            # Clean gene names
            gene_names = df["names"].astype(str).str.strip()

            genes = set(
                gene_names[values < threshold]
            )

            genes_per_celltype.append(genes)

            print(f"{sheet}: {len(genes)} significant genes")

        except:
            print("error")

    common_genes = set.intersection(*genes_per_celltype)

    print(f"\nGenes significant in ALL celltypes: {len(common_genes)}")

    return sorted(common_genes)

In [ ]:
genes = genes_significant_all_celltypes(
    "c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Downstream_processing/all_ranked_markers_summary.xlsx",
    experiment="CITEseq_Toxo_Test", # for Toxo, the file has added _WT and _Test for its 2 treatment groups
    metric="padj",
    threshold=0.30
)

print(genes)

# in conclusion, this does not really work, since no significant genes exist, which are significant in one experiment

GSEApy on the pseudobulk samples using DESeq2 data

In [ ]:
# GO_Biological_Process_2026
# GO_Cellular_Component_2026
# GO_Molecular_Function_2026
# KEGG_2026
# KEGG_2019_Mouse
# MSigDB_Hallmark_2020
# Mouse_Gene_Atlas
# Reactome_Pathways_2024
# WikiPathways_2024_Mouse

In [ ]:
results_toxo_vs_other  # this DE is toxo vs other immunogenic groups
# this has Toxo vs other group, so a positive value means enriched in Toxo, while a negative value means enriched in the other group

In [ ]:
experiment = "CITEseq_LNP_pIC_LNPs"
celltype = "Late Mature"

res = results_toxo_vs_other[(experiment, celltype)]

rnk = (
    res[["stat"]]
    .dropna()
    .sort_values("stat", ascending=False)
)

In [ ]:
# stat Wald test statistic (logFC divided by SE)
rnk = (
    res[["stat"]]
    .dropna()
    .sort_values("stat", ascending=False)
)

rnk.index.name = "gene"

In [ ]:
rnk # created a new column with "gene" as the first column

In [ ]:
pre_res = gp.prerank(
    rnk=rnk,
    gene_sets="GO_Biological_Process_2026",
    threads=8,
    min_size=15,
    max_size=500,
    permutation_num=1000,
    seed=42,
    outdir=None
)

In [ ]:
# extract results
gsea = pre_res.res2d

gsea[
    ["Term", "ES", "NES", "NOM p-val", "FDR q-val"]
].sort_values(["FDR q-val", "NES"], ascending=[True, False]).head(20)

NES > 0
→ gene set enriched among Toxo-upregulated genes (genes higher in toxo)

NES < 0
→ gene set enriched among reference-group-upregulated genes (genes higher in the reference)

In [ ]:
print(pre_res.res2d["Term"].head(20).to_list())

In [ ]:
term = 'Response to Type I Interferon (GO:0034340)'

gp.gseaplot(
    rank_metric=pre_res.ranking,
    term=term,
    **pre_res.results[term]
)

Automated version of the previous workflow (GSEApy)

In [ ]:
gsea_results = {}

for (experiment, celltype), res in results_toxo_vs_other.items():

    print(f"Running GSEA: {experiment} | {celltype}")

    # Create ranked gene list from DESeq2 Wald statistic
    rnk = (
        res[["stat"]]
        .dropna()
        .sort_values("stat", ascending=False)
    )

    # Remove duplicate genes if present
    rnk = rnk[~rnk.index.duplicated(keep="first")]

    # Skip if too few genes
    if len(rnk) < 20:
        print("  Skipping: too few genes")
        continue

    try:
        pre_res = gp.prerank(
            rnk=rnk,
            gene_sets="GO_Biological_Process_2026",
            threads=8,
            min_size=15,
            max_size=500,
            permutation_num=1000,
            seed=42,
            outdir=None
        )

        gsea_results[(experiment, celltype)] = pre_res

        print(f"  Done: {len(pre_res.res2d)} pathways")

    except Exception as e:
        print(f"  Failed: {e}")

In [ ]:
for result in gsea_results:
    print(f"result of f{result}")
    print(gsea_results[result])

In [ ]:
# # save the GSEA results for later
# import pickle

# gsea_results_df = {
#     key: result.res2d
#     for key, result in gsea_results.items()
# }

# with open("GSEA/gsea_results.pkl", "wb") as f:
#     pickle.dump(gsea_results_df, f)

In [ ]:
# to open them: 
with open("GSEA/gsea_results.pkl", "rb") as f:
    gsea_results = pickle.load(f)

In [ ]:
gsea_results

In [ ]:
# extract results
gsea = gsea_results[('CITEseq_LNP_CpG_LNPs',
  'Late Mature')]

gsea[
    ["Term", "ES", "NES", "NOM p-val", "FDR q-val"]
].sort_values(["NES", "FDR q-val"], ascending=[False, True]).head(20)

In [ ]:
with pd.ExcelWriter("GSEA/gsea_results.xlsx", engine="openpyxl") as writer:

    for (experiment, celltype), df in gsea_results.items():

        # Excel sheet names cannot contain these characters
        sheet_name = f"{experiment}_{celltype}"
        sheet_name = sheet_name.replace("/", "_").replace("\\", "_")
        sheet_name = sheet_name.replace(":", "_")
        sheet_name = sheet_name.replace("*", "_")
        sheet_name = sheet_name.replace("?", "_")
        sheet_name = sheet_name.replace("[", "_").replace("]", "_")

        # Excel has a 31-character sheet-name limit
        sheet_name = sheet_name[:31]

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        # make an excel of the gsea results with each sheet being a key of the dictionary

ORA analysis on the conserved and unique genes for the celltypes. ORA only needs a list of genes, no statistics, while GSEA needs a list of geen-level summary statistics. ORA results in a per-pathway hypergeometric test result. It is based on arbitrary thresholds and ignores any statistics. It assumes independence of genes and pathways.

We will do this with GSEApy, using the "enrichr" function. 

In [ ]:
adata # contains all genes, and X layer is lognormalized

In [ ]:
conserved_markers # holds the conserved markers per celltype, with some statistics

In [ ]:
celltypes

In [ ]:
# this is the function that looks through all genes for each celltype across all experimental groups
# specify the following in this order: celltype, max adjusted p-value, mean log fold change, amount of experiments in which the gene is significant
markers = find_markers_from_excel("Late Mature", 0.05, 1, 1)["names"].to_list() # 60% experiments for cutoff, for Late Immature, maybe lower to 0.5 logFC
# need 100-1000 genes preferably for ORA, 100-200 for UCell
markers

In [ ]:
print(gp.get_library_name(organism="Mouse"))

In [ ]:
enr = gp.enrichr(
    gene_list=markers,                              # or "./tests/data/gene_list.txt"
    gene_sets=['GO_Biological_Process_2026'], # 'GO_Biological_Process_2026', 
    organism='mouse',                                 # set to your organism, e.g. 'Yeast'
    outdir=None,                                       # keep results in memory only
)

In [ ]:
enr.results.head(50)

Find unique genes

In [ ]:
import re

# the excel file used for this contains a sheet per celltype, and then statistics for each gene in each experiment
# the statistics come from a wilcoxon rank test, which runs DE across celltype groups (because celltypes were specified)
 
def _to_numeric_safe(series):
    """Coerce to numeric, tolerating comma-decimal strings."""
    if series.dtype == object:
        series = series.astype(str).str.replace(",", ".", regex=False)
    return pd.to_numeric(series, errors="coerce")


# main function, find unique genes
def find_unique_genes(
    experiment,
    target_celltype,
    max_padj=0.05,
    min_logFC=1,
    require_experiment_unique=True,   # hard-filter genes also sig. in other experiments (same celltype)
):
    """
    Find genes significantly upregulated in target_celltype for a
    specific experiment, that are:
      (1) NOT significantly upregulated in any OTHER celltype for
          this same experiment, AND
      (2) NOT significantly upregulated in target_celltype under
          any OTHER experiment.

    Returns a DataFrame (not just a gene list) with:
      - celltype_uniqueness_score : target_logFC - strongest competing
        logFC among other celltypes (this experiment)
      - experiment_uniqueness_score : target_logFC - strongest competing
        logFC in target_celltype under other experiments
      - overall_uniqueness_score : min of the two above (weakest-link
        logic -- a gene is only as "unique" as its least unique axis)
    """

    path = (
        "c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/"
        "Lode/Downstream_processing/all_ranked_markers_summary.xlsx"
    )

    # define columns in the excel
    padj_col = f"padj_{experiment}"
    logFC_col = f"logFC_{experiment}"

    with pd.ExcelFile(path) as xls:

        if target_celltype not in xls.sheet_names:
            raise ValueError(
                f"'{target_celltype}' not found. "
                f"Available celltypes: {xls.sheet_names}"
            )

        # get to the sheet of the target celltype
        target_df = pd.read_excel(xls, sheet_name=target_celltype)
        missing = [c for c in (padj_col, logFC_col) if c not in target_df.columns]
        if missing:
            raise KeyError(
                f"Target celltype '{target_celltype}' is missing {missing} "
                f"for experiment '{experiment}'."
            )
        target_df = target_df.copy()

        # -------------------------
        # Detect all experiments present in target sheet
        # -------------------------
        all_padj_cols = [c for c in target_df.columns if re.match(r"^padj_", c)]
        all_experiments = [c[len("padj_"):] for c in all_padj_cols]
        other_experiments = [e for e in all_experiments if e != experiment]

        for c in target_df.columns:
            if c.startswith("padj_") or c.startswith("logFC_"):
                target_df[c] = _to_numeric_safe(target_df[c])

        # -------------------------
        # (1) Significant genes for target experiment, target celltype
        # -------------------------
        sig_mask = (
            (target_df[padj_col] <= max_padj) &
            (target_df[logFC_col] >= min_logFC)
        )
        target_sig = (
            target_df.loc[sig_mask, ["names", padj_col, logFC_col]]
            .dropna(subset=["names"])
            .rename(columns={padj_col: "target_padj", logFC_col: "target_logFC"})
        )

        # -------------------------
        # (1a) Cross-CELLTYPE comparison (same experiment)
        # -------------------------
        other_celltypes = [ct for ct in xls.sheet_names if ct != target_celltype]
        skipped_celltypes = []
        other_ct_records = []

        for ct in other_celltypes:
            df = pd.read_excel(xls, sheet_name=ct)
            if padj_col not in df.columns or logFC_col not in df.columns:
                skipped_celltypes.append(ct)
                continue
            sub = df[["names", padj_col, logFC_col]].dropna(subset=["names"]).copy()
            sub[padj_col] = _to_numeric_safe(sub[padj_col])
            sub[logFC_col] = _to_numeric_safe(sub[logFC_col])
            other_ct_records.append(sub.rename(columns={padj_col: "padj", logFC_col: "logFC"}))

        if skipped_celltypes:
            print(
                f"Note: experiment '{experiment}' not present in "
                f"{len(skipped_celltypes)} celltype sheet(s), skipped: {skipped_celltypes}"
            )

    other_ct_all = (
        pd.concat(other_ct_records, ignore_index=True) if other_ct_records
        else pd.DataFrame(columns=["names", "padj", "logFC"])
    )
    ct_stats = other_ct_all.groupby("names").agg(
        n_other_celltypes_tested=("logFC", "count"),
        max_other_celltype_logFC=("logFC", "max"),
    )
    other_celltype_sig_genes = set(
        other_ct_all.loc[
            (other_ct_all["padj"] <= max_padj) & (other_ct_all["logFC"] >= min_logFC),
            "names"
        ]
    )

    # -------------------------
    # (1b) Cross-EXPERIMENT comparison (same celltype)
    # -------------------------
    other_exp_records = []
    for other_exp in other_experiments:
        p_col, l_col = f"padj_{other_exp}", f"logFC_{other_exp}"
        sub = target_df[["names", p_col, l_col]].dropna(subset=["names"]).copy()
        sub = sub.rename(columns={p_col: "padj", l_col: "logFC"})
        other_exp_records.append(sub)

    other_exp_all = (
        pd.concat(other_exp_records, ignore_index=True) if other_exp_records
        else pd.DataFrame(columns=["names", "padj", "logFC"])
    )
    exp_stats = other_exp_all.groupby("names").agg(
        n_other_experiments_tested=("logFC", "count"),
        max_other_experiment_logFC=("logFC", "max"),
    )
    other_experiment_sig_genes = set(
        other_exp_all.loc[
            (other_exp_all["padj"] <= max_padj) & (other_exp_all["logFC"] >= min_logFC),
            "names"
        ]
    )

    # -------------------------
    # Combine
    # -------------------------
    result = target_sig.merge(ct_stats, left_on="names", right_index=True, how="left")
    result = result.merge(exp_stats, left_on="names", right_index=True, how="left")

    if require_experiment_unique:
        result = result[
            ~result["names"].isin(other_celltype_sig_genes) &
            ~result["names"].isin(other_experiment_sig_genes)
        ].copy()
    else:
        result = result[~result["names"].isin(other_celltype_sig_genes)].copy()

    result["celltype_uniqueness_score"] = (
        result["target_logFC"] - result["max_other_celltype_logFC"]
    ).fillna(result["target_logFC"])

    result["experiment_uniqueness_score"] = (
        result["target_logFC"] - result["max_other_experiment_logFC"]
    ).fillna(result["target_logFC"])

    result["overall_uniqueness_score"] = result[
        ["celltype_uniqueness_score", "experiment_uniqueness_score"]
    ].min(axis=1)

    result["n_other_celltypes_tested"] = result["n_other_celltypes_tested"].fillna(0).astype(int)
    result["n_other_experiments_tested"] = result["n_other_experiments_tested"].fillna(0).astype(int)

    result = result.sort_values("overall_uniqueness_score", ascending=False).reset_index(drop=True)

    return result

In [ ]:
unique_genes = find_unique_genes(
    experiment="CITEseq_Toxo_Test",
    target_celltype="Late Mature",
    max_padj=0.05,
    min_logFC=1
)

print(len(unique_genes))
print(unique_genes[:50])

In [ ]:
print(unique_genes["names"].to_list())